# Plots of Dirichlet regression models

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import scipy
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker

import seaborn as sns


import matplotlib.cm as cm
import matplotlib.colors as colrs
from matplotlib.patches import PathPatch
from matplotlib.path import Path


import matplotlib.cm as cm
import matplotlib.colors as colrs

import json
from shapely.geometry import shape as Shape, Polygon, mapping, box
from shapely.ops import transform as Shapely_transform

from scipy.stats import gaussian_kde

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error, mean_absolute_error

import folium
import mapply

from scipy.optimize import curve_fit

from tqdm import tqdm
from scipy import interpolate

In [ ]:
# Font
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['CMU Serif Roman'] + plt.rcParams['font.serif']
plt.rcParams['font.size'] = 16

mapply.init(
    n_workers=30,
    chunk_size=1,
    progressbar=True
)

## Load data

In [ ]:
YEAR = 2019
YEAR = 2024

import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f'{WORKING_DIR}/data'
IMG_DIR = f'{WORKING_DIR}/images'
REGRESSION_DIR = f'{DATA_DIR}/dirichlet/{YEAR}'

### Parties details

In [ ]:
fd = open(f'{DATA_DIR}/parties_description/political_parties_description_europe_{YEAR}.json', 'r')
political_parties_description = json.load(fd)
fd.close()

votes_columns = [f'{party}_votes' for party in political_parties_description]
votes_columns += ['others_votes']
parties_names = [political_parties_description[party]['name'] for party in political_parties_description]
parties_names += ['Others']
lr_scores = [political_parties_description[party]['lr_score'] for party in political_parties_description]

### Load election results

In [ ]:
df_communes_elections_traffic_social_urban = pd.read_pickle(f'{DATA_DIR}/df_communes_elections_traffic_social_urban_europe_{YEAR}.pkl')
df_communes_elections_traffic_social_urban.head(2)

### Load predictions

In [ ]:
model_names  = [
                'pop', 
                'unemployment', 
                'income', 
                'messaging',
                'streamming',
                'news',
                'social_media',
                'apps', 
                'income_unemployment_pop',
                'income_unemployment_pop_apps',
                ]

model_names_humans = [
                        'Population', 
                        'Unemployment', 
                        'Income', 
                        'Messaging',
                        'Streamming',
                        'News',
                        'Social Media',
                        'Mobile Services',
                        'Socioeconomic',
                        'All',
                        ]


model_names = model_names[::-1]
model_names_humans = model_names_humans[::-1]

df_predictions = {}
for model in tqdm(model_names):
    df_predictions[model] = pd.read_csv(f'{REGRESSION_DIR}/df_prediction_{model}.csv')
    df_predictions[model].rename(columns={
        'LREM.MoDem_votes': 'LREM-MoDem_votes',
        'LR.LC_votes': 'LR-LC_votes',
        'SP.PP.RDG.ND_votes': 'SP-PP-RDG-ND_votes'}, inplace=True)
    df_predictions[model] *= 100
    df_predictions[model]['insee'] = list(df_communes_elections_traffic_social_urban['insee'])

df_predictions[model].head(2)

### Load coefficients

In [ ]:
df_coefficient = pd.read_csv(f'{REGRESSION_DIR}/df_coefficients_{model}.csv')
df_coefficient

In [ ]:
df_coefficients = {}
for model in tqdm(model_names):
    df_coefficient = pd.read_csv(f'{REGRESSION_DIR}/df_coefficients_{model}.csv')
    df_std_error = pd.read_csv(f'{REGRESSION_DIR}/df_coefficients_std_error_{model}.csv')


    df_coefficient.rename(columns={'Unnamed: 0': 'variable', 'coefficients': 'coefficient'}, inplace=True)
    df_coefficient['model'] = model
    df_coefficient['party'] = df_coefficient.apply(lambda row: row['variable'].replace('_votes', '').split(':')[0], axis=1)
    df_coefficient['party'] = df_coefficient['party'].apply(lambda party: 'LREM-MoDem' if 'LREM.MoDem' in party else party)
    df_coefficient['party'] = df_coefficient['party'].apply(lambda party: 'LR-LC' if 'LR.LC' in party else party)
    df_coefficient['party'] = df_coefficient['party'].apply(lambda party: 'SP-PP-RDG-ND' if 'SP.PP.RDG.ND' in party else party)

    df_coefficient['variable'] = df_coefficient.apply(lambda row: row['variable'].split(':')[1], axis=1)
    df_coefficients[model] = df_coefficient

    df_std_error.rename(columns={'Unnamed: 0': 'variable', 'x': 'std_error'}, inplace=True)
    df_std_error['model'] = model
    df_std_error['party'] = df_std_error.apply(lambda row: row['variable'].replace('_votes', '').split(':')[0], axis=1)
    df_std_error['party'] = df_std_error['party'].apply(lambda party: 'LREM-MoDem' if 'LREM.MoDem' in party else party)
    df_std_error['party'] = df_std_error['party'].apply(lambda party: 'LR-LC' if 'LR.LC' in party else party)
    df_std_error['party'] = df_std_error['party'].apply(lambda party: 'SP-PP-RDG-ND' if 'SP.PP.RDG.ND' in party else party)

    df_std_error['variable'] = df_std_error.apply(lambda row: row['variable'].split(':')[1], axis=1)
    
    df_coefficient = df_coefficient.merge(df_std_error, on=['model', 'party', 'variable'], how='inner')
    df_coefficient = df_coefficient[['model', 'party', 'variable', 'coefficient', 'std_error']]
    df_coefficients[model] = df_coefficient

df_coefficients['income_unemployment_pop'].head(10)

### Rename features variables

In [ ]:

variables_human = {
    'median_income': 'Median Income',
    'unemployment_ratio': 'Unemployment',
    'pop_0_14': 'Pop 0-14',
    'pop_15_29': 'Pop 15-29',
    'pop_30_44': 'Pop 30-44',
    'pop_45_59': 'Pop 45-59',
    'pop_60_74': 'Pop 60-74',
    'pop_75_89': 'Pop 75-89',
    'pop_90': 'Pop 90+',
    'Amazon.Prime.Video_srca' : 'Amazon Prime Video',
    'Apple.Music_srca' : 'Apple Music',
    'Apple.TV_srca' : 'Apple TV',
    'Apple.Video_srca' : 'Apple Video',
    'Apple.iMessage_srca' : 'Apple iMessage',
    'Betting_srca': 'Betting',
    'CanalPlus_srca': 'Canal+',
    'DailyMotion_srca': 'Daily Motion',
    'Deezer_srca': 'Deezer',
    'Discord_srca': 'Discord',
    'Disney+_srca': 'Disney+',
    'Disney._srca': 'Disney+',
    'Facebook_srca': 'Facebook',
    'Finances_srca': 'Finances',
    'Google.News_srca': 'Google News',
    'Instagram_srca': 'Instagram',
    'LinkedIn_srca': 'LinkedIn',
    'Money_Stock_srca': 'Money stock',
    'Molotov.TV_srca': 'Molotov TV',
    'Netflix_srca': 'Netflix',
    'News_srca': 'News websites',
    'NewsMag_srca': 'News magazines',
    'NewsPaper_srca': 'NewsPaper',
    'Pluto.TV_srca': 'Pluto TV',
    'Signal_srca': 'Signal',
    'Sports.News_srca': 'Sports news',
    'SnapChat_srca': 'Snapchat',
    'Sports_News_srca': 'Sports news websites',
    'Spotify_srca': 'Spotify',
    'TV5MONDE_srca': 'TV5MONDE',
    'Telegram_srca': 'Telegram',
    'TikTok_srca': 'TikTok',
    'Twitch_srca': 'Twitch',
    'Twitter_srca': 'Twitter',
    'Vine_srca': 'Vine',
    'WhatsApp_srca': 'WhatsApp',
    'Wikipedia_srca': 'Wikipedia',
    'Youtube_srca': 'Youtube',
    '(Intercept)': 'intercept',
}


for model in tqdm(model_names):
    df_coefficients[model]['variable'] = df_coefficients[model]['variable'].apply(lambda variable: variables_human[variable])

df_coefficients['income_unemployment_pop'].head(2)

### Load Parameters and Log-likelihood

## Plots

### Accuracy | R2 score

In [ ]:
r2_parties = np.zeros((len(df_predictions), len(votes_columns)))
for i, (model_name, df_prediction) in enumerate(df_predictions.items()):
    for j, party in enumerate(votes_columns):
        r2 = r2_score(df_communes_elections_traffic_social_urban[party], df_prediction[party])

        n = df_prediction.shape[0]
        k = df_coefficients[model_name].shape[0] - (7 if YEAR == 2019 else 8)
        r2_score_adjusted = 1 - ((1 - r2) * (n - 1)) / (n - k - 1)    
        r2_parties[i, j] = r2_score_adjusted

r2_parties[r2_parties < 0] = 0

r2_parties = r2_parties**.5

In [ ]:

fig = plt.figure(figsize=(15, 15))

my_cmap = matplotlib.colormaps['GnBu'].copy()
my_norm = colrs.Normalize(vmin=0, vmax=1)

heatmap = sns.heatmap(r2_parties[:,:-1], cmap=my_cmap, norm=my_norm, annot=True, cbar=False, fmt='.2f', 
                      annot_kws={'fontsize':30})

xticks_labels = [f'{party_name}' for party_name, lr_score in zip(parties_names, lr_scores)]
heatmap.set_xticklabels(xticks_labels, fontsize=30, rotation=35, ha='right')

plt.yticks(np.arange(len(df_predictions))+.5, model_names_humans, fontsize=30, rotation=0)
if YEAR == 2024:
    plt.yticks([])





plt.show()

In [ ]:
df_r2 = pd.DataFrame(r2_parties[:,:-1], columns=parties_names[:-1], index=model_names_humans)
df_r2.to_pickle(f'{DATA_DIR}/df_r2_scores_{YEAR}.pkl')

In [ ]:
fig = plt.figure(figsize=(6, 1))

my_cmap = matplotlib.colormaps['GnBu'].copy()
my_norm = colrs.Normalize(vmin=0, vmax=1)

ax = fig.add_axes([0, 0, .7, .15])
sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
sm.set_array([])
clb = plt.colorbar(sm, cax=ax, orientation='horizontal')
clb.ax.xaxis.set_ticks_position('default')
clb.ax.set_title(rf'Correlation $(\rho)$', fontsize=18)
clb.ax.tick_params(labelsize=18)

plt.savefig(f'{IMG_DIR}/dirichlet_regression/colorbar.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig = plt.figure(figsize=(1, 6))


my_cmap = matplotlib.colormaps['Reds'].copy()
my_cmap.set_bad(color='w')
my_norm = colrs.Normalize(vmin=0, vmax=1)


ax = fig.add_axes([0, 0, .15, .7])
sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
sm.set_array([])
clb = plt.colorbar(sm, cax=ax, orientation='vertical')
clb.ax.xaxis.set_ticks_position('default')
clb.ax.set_title(rf'$\rho$', fontsize=18, pad=20, loc='left')
clb.ax.tick_params(labelsize=18)

plt.savefig(f'{IMG_DIR}/dirichlet_regression/colorbar_vertical_rho.pdf', bbox_inches='tight')
plt.show()

### Gain

In [ ]:
all = r2_parties[0][:-1]
mean_all = np.mean(all)
print(np.mean(all))

socio = r2_parties[1][:-1]
mean_socio = np.mean(socio)
print(np.mean(socio))


print((mean_all - mean_socio) / mean_socio * 100)
print(np.mean( (all - socio) / socio ) * 100)

In [ ]:
(0.725572 - 0.57714)/ 0.57714 * 100

In [ ]:
row_0 = r2_parties[0][:-1]
row_1 = r2_parties[1][:-1]


relative_gain = (row_0 - row_1) / row_1 * 100
relative_gain = np.round(relative_gain, 2)
for party, gain in zip(parties_names[:-1], relative_gain):
    print(f'{party}: {gain}%')

relative_gain = list(relative_gain)
print('---')
print(f'The mean relative gain is {np.round(np.mean(relative_gain),2)}%')

if YEAR == 2019:
    print(f'The mean relative gain is {np.round(np.mean(relative_gain[:2]+relative_gain[3:]),2)}%')

In [ ]:
if YEAR == 2019:

    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(15, 10), gridspec_kw={'height_ratios': [1, 3]})
    fig.subplots_adjust(hspace=0.05)  # adjust space between axes

    ax1.grid(axis='y', alpha=0.1)
    ax2.grid(axis='y', alpha=0.1)
    color = 'tab:blue'

    for index, (party, gain) in enumerate(zip(parties_names[:-1], relative_gain)):
        ax1.bar(party, gain, color=color, alpha=0.7)
        ax1.text(index, gain + 1, f'{gain:.2f}%', ha='center', va='bottom', fontsize=25)
    
    for index, (party, gain) in enumerate(zip(parties_names[:-1], relative_gain)):
        ax2.bar(party, gain, color=color, alpha=0.7)
        if index != 2:
            ax2.text(index, gain + 1, f'{gain:.2f}%', ha='center', va='bottom', fontsize=25)

    ax2.set_ylim(0, 55)  # most of the data
    ax1.set_ylim(249, 370)  # outliers only
    # hide the spines between ax and ax2
    ax1.spines.bottom.set_visible(False)
    ax2.spines.top.set_visible(False)

    ax1.xaxis.tick_top()
    ax1.set_xticks([])
    ax1.tick_params(labeltop=False)  # don't put tick labels at the top


    ax2.set_xticks(np.arange(len(parties_names[:-1])))
    ax2.set_xticklabels(parties_names[:-1])

    d = .85  # proportion of vertical to horizontal extent of the slanted line
    kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
                linestyle="none", color='k', mec='k', mew=1, clip_on=False)
    ax1.plot([0, 1], [0, 0], transform=ax1.transAxes, **kwargs)
    ax2.plot([0, 1], [1, 1], transform=ax2.transAxes, **kwargs)

    plt.xticks(rotation=35, ha='right', fontsize=30)
    ax2.set_ylabel('Relative Gain (%)', loc='top', fontsize=30, labelpad=20)

    ax1.set_yticks([250, 300, 350])
    ax1.set_yticklabels(['250', '300', '350'], fontsize=30)
    ax2.set_yticks([0, 10, 20, 30, 40, 50])
    ax2.set_yticklabels(['0', '10', '20', '30', '40', '50'], fontsize=30)
    
    plt.axhline(0, color='k', linestyle='--', linewidth=1)  
    plt.savefig(f'{IMG_DIR}/dirichlet_regression/relative_gain_adj_r2_europe_{YEAR}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
if YEAR == 2024:
    fig = plt.figure(figsize=(15, 10))

    plt.grid(axis='y', alpha=0.1)
    
    color = 'tab:green'

    for index, (party, gain) in enumerate(zip(parties_names[:-1], relative_gain)):
        plt.bar(party, gain, color=color, alpha=0.7)
        plt.text(index, gain + 1, f'{gain:.2f}%', ha='center', va='bottom', fontsize=25)


    plt.ylim(0, 200)
    plt.xticks(rotation=35, ha='right', fontsize=30)
    plt.ylabel('Relative Gain (%)', fontsize=30, labelpad=20)
    plt.axhline(0, color='k', linestyle='--', linewidth=1)  
    plt.yticks(fontsize=30)
    plt.savefig(f'{IMG_DIR}/dirichlet_regression/relative_gain_adj_r2_europe_{YEAR}.pdf', bbox_inches='tight')
    plt.show()

In [ ]:




# r2_parties[r2_parties < 0] = 0

row_0 = r2_parties[0][:-1]
row_1 = r2_parties[1][:-1]

relative_gain = (row_0 - row_1) / row_1 * 100
relative_gain = np.round(relative_gain, 2)
for party, gain in zip(parties_names[:-1], relative_gain):
    print(f'{party}: {gain}%')

relative_gain = list(relative_gain)
print('---')
print(f'The mean relative gain is {np.round(np.mean(relative_gain),2)}%')

if YEAR == 2019:
    print(f'The mean relative gain is {np.round(np.mean(relative_gain[:2]+relative_gain[3:]),2)}%')

if YEAR == 2019:
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(15, 20), gridspec_kw={'height_ratios': [2, 3]})
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(15, 20), gridspec_kw={'height_ratios': [2, 3]})

fig.subplots_adjust(hspace=0.05)  # adjust space between axes


color = 'tab:grey'



for index, (party, gain) in enumerate(zip(parties_names[:-1], relative_gain)):
    ax1.bar(index+.5, gain, color=color, alpha=0.5)
    ax1.text(index+.5, gain + 1, f'{gain:.2f}%', ha='center', va='bottom', fontsize=30)

ax1.set_ylim(0, 120)
yticks = np.arange(0, 120, 20)


if YEAR == 2019:
    ax1.set_yticks(yticks)
    ax1.set_yticklabels([f'{y}' for y in yticks], fontsize=30)
    ax1.set_ylabel('Relative Gain (%)', fontsize=30, labelpad=20, loc='center')
else:
    ax1.set_yticks(yticks)
    ax1.set_yticklabels([])
ax1.grid(axis='y', alpha=0.2)

my_cmap = matplotlib.colormaps['GnBu'].copy()
my_norm = colrs.Normalize(vmin=0, vmax=1)

my_cmap = matplotlib.colormaps['RdBu_r'].copy()
my_cmap.set_bad(color='w')
my_norm = colrs.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

heatmap = sns.heatmap(r2_parties[:,:-1], cmap=my_cmap, norm=my_norm, annot=True, cbar=False, fmt='.2f', 
                      annot_kws={'fontsize':30}, ax=ax2)

xticks_labels = [f'{party_name}' for party_name, lr_score in zip(parties_names, lr_scores)]
heatmap.set_xticklabels(xticks_labels, fontsize=30, rotation=35, ha='right')



plt.yticks(np.arange(len(df_predictions))+.5, model_names_humans, fontsize=30, rotation=0)
if YEAR == 2024:
    plt.yticks([])

plt.savefig(f'{IMG_DIR}/dirichlet_regression/rho_relative_gain_europe_{YEAR}.pdf', bbox_inches='tight', transparent=True)

plt.show()

### Spatial plots of predictions

In [ ]:
def PolygonPatch(polygon, **kwargs):
    """Replaces descartes.PolygonPatch for Shapely 2.0+"""
    # Get the exterior path codes and coordinates
    ext_coords = np.asarray(polygon.exterior.coords)
    codes = [Path.MOVETO] + [Path.LINETO] * (len(ext_coords) - 2) + [Path.CLOSEPOLY]
    vertices = ext_coords

    # Add any interior holes if they exist
    for interior in polygon.interiors:
        int_coords = np.asarray(interior.coords)
        int_codes = (
            [Path.MOVETO] + [Path.LINETO] * (len(int_coords) - 2) + [Path.CLOSEPOLY]
        )
        vertices = np.concatenate([vertices, int_coords])
        codes.extend(int_codes)

    path = Path(vertices, codes)
    return PathPatch(path, **kwargs)


def PolygonPatch(polygon, **kwargs):
    """Replaces descartes.PolygonPatch for Shapely 2.0+ (Handles MultiPolygons)"""
    # If it's a MultiPolygon, break it down into individual Polygons
    if hasattr(polygon, "geoms"):
        polygons = polygon.geoms
    else:
        polygons = [polygon]

    vertices_list = []
    codes_list = []

    for poly in polygons:
        # Process individual polygon exterior
        ext_coords = np.asarray(poly.exterior.coords)
        ext_codes = (
            [Path.MOVETO] + [Path.LINETO] * (len(ext_coords) - 2) + [Path.CLOSEPOLY]
        )

        vertices_list.append(ext_coords)
        codes_list.extend(ext_codes)

        # Process individual polygon interior holes
        for interior in poly.interiors:
            int_coords = np.asarray(interior.coords)
            int_codes = (
                [Path.MOVETO] + [Path.LINETO] * (len(int_coords) - 2) + [Path.CLOSEPOLY]
            )

            vertices_list.append(int_coords)
            codes_list.extend(int_codes)

    # Combine everything into a single Matplotlib path
    if vertices_list:
        vertices = np.concatenate(vertices_list)
        path = Path(vertices, codes_list)
        return PathPatch(path, **kwargs)
    else:
        raise ValueError("Provided geometry contains no valid polygons.")

In [ ]:
def plot_prediction(region, fig_ratio, shapes, values, my_cmap, my_norm, filename, title=None, show=True):
    
    fig = plt.figure(figsize=fig_ratio)

    ax = fig.add_axes([0, 0, 1, .95])
    ax.set_rasterized(True)

    region_ = Shapely_transform(lambda x, y: (y, x), region)

    for shape, value in zip(shapes, values):
        interception_area = shape.intersection(region_).area
        if interception_area/shape.area < .9: # skip shapes that are less than 90% inside the region
            continue
        
        color = colrs.to_hex(my_cmap(my_norm(value)))
        shape = Shapely_transform(lambda x, y: (y, x), shape)

        patch = PolygonPatch(shape, fc=color, ec='lightgrey')
        ax.add_patch(patch)

        
    bounds = region.bounds
    plt.xlim(bounds[0], bounds[2])
    plt.ylim(bounds[1], bounds[3])
    plt.axis('off')
    plt.autoscale()

    plt.savefig(f'{IMG_DIR}/dirichlet_regression/maps/{filename}.pdf', bbox_inches='tight', dpi=200)
    if show:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
cmap = 'RdYlGn_r'

vmin = 5
vmax = 50
vcenter = round((vmin + vmax) / 2)

my_cmap = cm.get_cmap(cmap)
my_norm = colrs.TwoSlopeNorm(vmin = vmin,
                            vcenter = vcenter,
                            vmax = vmax)

In [ ]:
fig = plt.figure(figsize=(6, 1))

ax = fig.add_axes([0, 0, .7, .15])
sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
sm.set_array([])

clb = plt.colorbar(sm, cax=ax, orientation='horizontal')
clb.ax.set_xticks([vmin, vcenter, vmax])
clb.ax.set_xticklabels([f'{vmin}%', f'{vcenter}%', f'{vmax}%'])

clb.ax.set_title('Votes', fontsize=18)
clb.ax.xaxis.set_ticks_position('default')
clb.ax.tick_params(labelsize=16)

plt.savefig(f'{IMG_DIR}/dirichlet_regression/maps/colorbar.pdf', bbox_inches='tight', dpi=200, transparent=True)
plt.show()

In [ ]:
party.split('_')[0]

In [ ]:
cities = ['Paris',]
fig_ratios = [(6, 7)]

df_selected_communes = df_communes_elections_traffic_social_urban[df_communes_elections_traffic_social_urban['urbanization_level'] != 'rural']
shapes = list(df_selected_communes['geometry'])

if YEAR == 2019:
    party = 'Ren_coal_votes'
else:
    party = 'RN_votes'
    party = 'LFI_votes'


party_name = party.split('_')
party_name = party_name[0] if len(party_name) == 2 else f'{party_name[0]}_{party_name[1]}'
party_name = political_parties_description[party_name]['name']
ground_truth = df_communes_elections_traffic_social_urban[party]
prediction_socioeconomic = df_predictions['income_unemployment_pop'][party]
prediction_socioeconomic_apps = df_predictions['income_unemployment_pop_apps'][party]
prediction_apps = df_predictions['apps'][party]
prediction_social_media = df_predictions['social_media'][party]
prediction_messaging = df_predictions['messaging'][party]
prediction_streamming = df_predictions['streamming'][party]
prediction_news = df_predictions['news'][party]




    
    
    

    
    
    



### Coefficients

In [ ]:
selected_model = 'income_unemployment_pop_apps'

df_coefficient_model = df_coefficients[selected_model].copy()

variables_coefficients = {}
for party in political_parties_description:
    df_coefficient_model_party = df_coefficient_model[df_coefficient_model['party'] == party].copy()

    variables = list(df_coefficient_model_party['variable'])
    coefficients = list(df_coefficient_model_party['coefficient'])
    std_errors = list(df_coefficient_model_party['std_error'])

    for variable, coefficient, std_error in zip(variables, coefficients, std_errors):
        if variable not in variables_coefficients:
            variables_coefficients[variable] = {
                                                'coefficients': [], 
                                                'coefficients_raw': [],
                                                'confidence_interval': [], 
                                                'significant': [],
                                                'std_error_interval': [],
                                                'sorted': []
                                            }
        
        variables_coefficients[variable]['coefficients_raw'].append(coefficient)
        

        # significant = (confidence_interval[0] <= 1 and confidence_interval[1] <= 1) or \
        #             (confidence_interval[0] >= 1 and confidence_interval[1] >= 1)

        confidence_interval = coefficient + np.array([-1, 1]) * 1.96 * np.array(std_error)
        std_error = coefficient + np.array([-1, 1])*np.array(std_error)
        coefficient = coefficient

        significant = (confidence_interval[0] <= 0 and confidence_interval[1] <= 0) or \
                    (confidence_interval[0] >= 0 and confidence_interval[1] >= 0)

        ci_lower = coefficient - confidence_interval[0]
        ci_upper = confidence_interval[1] - coefficient
        confidence_interval = np.array([[ci_lower], [ci_upper]])


        std_error_lower = coefficient - std_error[0]
        std_error_upper = std_error[1] - coefficient
        std_error_interval = np.array([[std_error_lower], [std_error_upper]])

        variables_coefficients[variable]['coefficients'].append(coefficient)
        variables_coefficients[variable]['confidence_interval'].append(confidence_interval)
        variables_coefficients[variable]['significant'].append(significant)
        variables_coefficients[variable]['std_error_interval'].append(std_error_interval)

del variables_coefficients['intercept']

variables_social = ['Median Income', 'Unemployment', 
                    'Pop 0-14','Pop 15-29', 'Pop 30-44', 'Pop 45-59', 'Pop 60-74'][::-1]
variables_traffic = set(variables_coefficients.keys()) - set(variables_social)

variables_coefficients_social = {variable:variables_coefficients[variable] for variable in variables_social}
variables_coefficients_traffic = {variable:variables_coefficients[variable] for variable in variables_traffic}

In [ ]:
variables_coefficients_traffic = {k:v for k,v in sorted(variables_coefficients_traffic.items(), key=lambda x: np.median(x[1]['coefficients']))}
variables_coefficients_traffic = {k:v for k,v in sorted(variables_coefficients_traffic.items(), key=lambda x: np.std(x[1]['coefficients']))}

In [ ]:
variables_order_2024 = ['TV5MONDE',
 'Pluto TV',
 'Google News',
 'Apple Video',
 'Disney+',
 'Apple iMessage',
 'Molotov TV',
 'News magazines',
 'Daily Motion',
 'Canal+',
 'Signal',
 'Discord',
 'Telegram',
 'TikTok',
 'Netflix',
 'Sports news',
 'Twitch',
 'Spotify',
 'Snapchat',
 'LinkedIn',
 'Apple Music',
 'Youtube',
 'WhatsApp',
 'Twitter',
 'NewsPaper',
 'Instagram',
 'Facebook']

In [ ]:
xleft = -.4
xright = .4
x_delta = xright - xleft

if YEAR == 2019:
    fig = plt.figure(figsize=(12, 20))
else:
    fig = plt.figure(figsize=(12, 20))


height = 100/(len(variables_coefficients_social)+len(variables_order_2024)) / 100
block_height_social = height * len(variables_coefficients_social)
ax = fig.add_axes([0, 0, 1, block_height_social])

# add stripes
for variable_index, variable in enumerate(variables_coefficients_social):
    if variable_index % 2 == 0:
        rect = plt.Rectangle((xleft, variable_index-.5), x_delta, 1, facecolor="tab:grey", edgecolor="tab:grey", alpha=0.05)
        ax.add_patch(rect)

for party_index, party in enumerate(political_parties_description):

    symbol = political_parties_description[party]['symbol']
    party_human_name = political_parties_description[party]['name']
    party_color = political_parties_description[party]['color']
    
    for variable_index, variable in enumerate(variables_coefficients_social):
        coefs = variables_coefficients_social[variable]['coefficients']
        confidence_intervals = variables_coefficients_social[variable]['confidence_interval']
        significants = variables_coefficients_social[variable]['significant']

        coef = coefs[party_index]
        confidence_interval = confidence_intervals[party_index]
        significant = significants[party_index]

        if YEAR == 2019:
            variable_index = variable_index - (0.1*3) + party_index*0.1
        else:
            variable_index = variable_index - (0.1*3) + party_index*0.1


        if significant:
            plt.scatter(coef, variable_index, s=150, color=party_color, alpha=1, marker=symbol, facecolors=party_color)
            plt.errorbar(coef, variable_index, xerr=confidence_interval, color=party_color, alpha=0.25, capsize=2, capthick=2)
        else:
            plt.scatter(coef, variable_index, s=150, color=party_color, alpha=0.1, marker=symbol, facecolors='None')
            plt.errorbar(coef, variable_index, xerr=confidence_interval, color=party_color, alpha=0.1, capsize=2, capthick=2)


indexs = range(len(variables_coefficients_social))
labels = []
for variable_index, variable in enumerate(variables_coefficients_social):
    variable_human = variable.replace('_', ' ').title()
    labels.append(f'{variable_human}')

if YEAR == 2019:
    plt.yticks(indexs, labels, fontsize=30)
else:
    plt.yticks([])

plt.axvline(0, color='grey', alpha=0.5, linestyle='--')
plt.xlim(xleft, xright)
plt.ylim(-.5, len(variables_coefficients_social)-.5)
plt.xlabel('Coefficient', fontsize=30)
plt.xticks(fontsize=30)

block_height = height * len(variables_order_2024)
ax = fig.add_axes([0, block_height_social+height, 1, block_height])

# add stripes
for variable_index, variable in enumerate(variables_order_2024):
    if variable_index % 2 == 0:
        rect = plt.Rectangle((xleft, variable_index-.5), x_delta, 1, facecolor="tab:grey", edgecolor="tab:grey", alpha=0.05)
        ax.add_patch(rect)


for party_index, party in enumerate(political_parties_description):
    if party == 'others':
        continue

    lr_score = political_parties_description[party]['lr_score']

    symbol = political_parties_description[party]['symbol']
    party_human_name = political_parties_description[party]['name']
    party_color = political_parties_description[party]['color']

    for variable_index, variable in enumerate(variables_order_2024):
        if variable not in variables_coefficients_traffic:
            continue

        coefs = variables_coefficients_traffic[variable]['coefficients']
        confidence_intervals = variables_coefficients_traffic[variable]['confidence_interval']
        significants = variables_coefficients_traffic[variable]['significant']

        coef = coefs[party_index]
        confidence_interval = confidence_intervals[party_index]
        significant = significants[party_index]

        if YEAR == 2019:
            variable_index = variable_index - (0.1*3) + party_index*0.1
        else:
            variable_index = variable_index - (0.1*3) + party_index*0.1
    
        
        if significant:
            plt.scatter(coef, variable_index, s=150, color=party_color, alpha=1, marker=symbol, facecolors=party_color)
            plt.errorbar(coef, variable_index, xerr=confidence_interval, color=party_color, alpha=0.5, capsize=2, capthick=2)
        else:
            plt.scatter(coef, variable_index, s=150, color=party_color, alpha=0.2, marker=symbol, facecolors='None')
            plt.errorbar(coef, variable_index, xerr=confidence_interval, color=party_color, alpha=0.2, capsize=2, capthick=2)
    

indexs = range(len(variables_order_2024))
labels = []
for variable_index, variable in enumerate(variables_order_2024):
    variable_human = variable.replace('_', ' ').replace(' srca', '').replace('generic', '')
    labels.append(f'{variable_human}')

labels = [label if label != 'NewsPaper' else 'Online news' for label in labels]

if YEAR == 2019:
    plt.yticks(indexs, labels, fontsize=30)
else:
    plt.yticks([])


plt.axvline(0, color='grey', alpha=0.5, linestyle='--')
plt.ylim(-.5, len(variables_order_2024)-.5)
plt.xticks(fontsize=30)
plt.xlim(xleft, xright)
plt.xticks([])

plt.savefig(f'{IMG_DIR}/dirichlet_regression/coefficients_{YEAR}.pdf', dpi=300, bbox_inches='tight', transparent=True)

plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 4))

for party_index, party in enumerate(political_parties_description):
    if party == 'others':
        continue

    lr_score = political_parties_description[party]['lr_score']

    symbol = political_parties_description[party]['symbol']
    party_human_name = political_parties_description[party]['name']
    party_color = political_parties_description[party]['color']

    # legend
    plt.scatter([], [], s=150, color=party_color, 
                marker=symbol, facecolors=party_color, 
                label=f'{party_human_name}')

plt.legend(fontsize=30, frameon=False, ncol=2)
plt.axis('off')
plt.savefig(f'{IMG_DIR}/dirichlet_regression/coefficients_{YEAR}_legend.pdf', dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:


# # ========== SOCIAL ==========




    


            















